In [1]:
"""Multivariate analysis (PCA and K-means) of the cleaned KITTI sensor data."""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = Path("data/cleaned/kitti.parquet")
OUTPUT_DIR = Path("eda")
PLOTS_DIR = OUTPUT_DIR / "plots"

RANDOM_STATE = 42
PCA_VARIANCE_TARGET = 0.95
CLUSTER_COUNTS = [3, 4, 5]
SILHOUETTE_MAX_SAMPLES = 10_000
MAX_LEGEND_DRIVES = 15

DRIVE_COL = "drive"
ORDER_COL = "frame"
X_COL = "east"
Y_COL = "north"

# Columns that identify rows rather than describe sensor behaviour.
EXCLUDE_COLS = [
    DRIVE_COL,
    ORDER_COL,
    "timestamp",
    "time",
    "filename",
    "file",
    "sequence",
    "sequence_id",
]

# Set to True to cluster on sensor behaviour only, so that the clusters are not
# driven by where the vehicle happens to be.
EXCLUDE_POSITION_FEATURES = False
POSITION_COLS = ["lat", "lon", "alt", "east", "north", "up"]

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [3]:
def save_figure(fig, name):
    """Save a figure into PLOTS_DIR."""
    fig.savefig(PLOTS_DIR / name, dpi=200, bbox_inches="tight")


def select_feature_columns(frame):
    """Return numeric, non-constant columns that are not identifiers."""
    excluded = set(EXCLUDE_COLS)
    if EXCLUDE_POSITION_FEATURES:
        excluded.update(POSITION_COLS)

    numeric_cols = frame.select_dtypes(include=np.number).columns
    return [
        col
        for col in numeric_cols
        if col not in excluded and frame[col].nunique(dropna=True) > 1
    ]


def count_components_for_variance(cumulative_variance, target):
    """Smallest number of components whose cumulative variance reaches target."""
    index = int(np.searchsorted(cumulative_variance, target))
    return min(index + 1, len(cumulative_variance))

In [4]:
df = pd.read_parquet(DATA_PATH)
analysis_df = df.reset_index(drop=True)

feature_cols = select_feature_columns(analysis_df)
if not feature_cols:
    raise ValueError("No usable numeric features found. Check your column names.")

print(f"Number of features: {len(feature_cols)}")
display(pd.DataFrame({"feature": feature_cols}))

Number of features: 38


,feature
0,lat
1,lon
2,alt
3,roll
4,pitch
5,yaw
6,vn
7,ve
8,vf
9,vl


In [5]:
X_raw = analysis_df[feature_cols]

X_imputed = SimpleImputer(strategy="median").fit_transform(X_raw)
X_scaled = StandardScaler().fit_transform(X_imputed)

print(f"Original matrix shape: {X_raw.shape}")
print(f"Prepared matrix shape: {X_scaled.shape}")
print(f"Missing values remaining: {int(np.isnan(X_scaled).sum())}")

Original matrix shape: (1436, 38)
Prepared matrix shape: (1436, 38)
Missing values remaining: 0


In [6]:
pca_full = PCA(svd_solver="full").fit(X_scaled)
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

n_components = count_components_for_variance(cumulative_variance, PCA_VARIANCE_TARGET)
if n_components < 2:
    raise ValueError("At least two PCA components are required for the plots below.")

pca = PCA(n_components=n_components, svd_solver="full", random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
pc_cols = [f"PC{i + 1}" for i in range(n_components)]

print(f"Original feature count: {len(feature_cols)}")
print(f"Components retained: {n_components}")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.2%}")

Original feature count: 38
Components retained: 14
Variance retained: 95.17%


In [7]:
components = np.arange(1, len(explained_variance) + 1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(components, explained_variance * 100, alpha=0.75, label="Individual variance")
ax.plot(components, cumulative_variance * 100, marker="o", label="Cumulative variance")
ax.axhline(
    PCA_VARIANCE_TARGET * 100,
    linestyle="--",
    label=f"{PCA_VARIANCE_TARGET:.0%} target",
)
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance (%)")
ax.set_title("PCA Explained Variance")
ax.set_xticks(components)
ax.legend()
fig.tight_layout()
save_figure(fig, "pca_explained_variance.png")
plt.show()

/var/folders/nk/l5mpcflj3z5396syzzlv9tw00000gn/T/ipykernel_64860/3085142020.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
pca_plot_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
has_drive = DRIVE_COL in analysis_df.columns

fig, ax = plt.subplots(figsize=(11, 7) if has_drive else (9, 7))
if has_drive:
    pca_plot_df[DRIVE_COL] = analysis_df[DRIVE_COL].fillna("Unknown").astype(str)
    sns.scatterplot(
        data=pca_plot_df,
        x="PC1",
        y="PC2",
        hue=DRIVE_COL,
        palette="tab20",
        s=25,
        alpha=0.7,
        linewidth=0,
        ax=ax,
    )
    if pca_plot_df[DRIVE_COL].nunique() > MAX_LEGEND_DRIVES:
        ax.legend(title="Drive", bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    ax.scatter(pca_plot_df["PC1"], pca_plot_df["PC2"], s=25, alpha=0.7)

ax.set_xlabel(f"PC1 ({explained_variance[0]:.2%})")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.2%})")
ax.set_title("PCA Projection Coloured by Drive")
fig.tight_layout()
save_figure(fig, "pca_pc1_pc2_by_drive.png")
plt.show()

/var/folders/nk/l5mpcflj3z5396syzzlv9tw00000gn/T/ipykernel_64860/3977798664.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
loadings = pd.DataFrame(pca.components_.T, index=feature_cols, columns=pc_cols)
display(loadings.round(4))

for pc in ["PC1", "PC2"]:
    print(f"\nLargest absolute loadings for {pc}:")
    display(
        loadings[pc]
        .abs()
        .sort_values(ascending=False)
        .head(10)
        .to_frame("absolute_loading")
    )

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14
lat,0.2353,-0.0412,0.0837,-0.2633,0.1347,0.0100,0.1083,0.0453,-0.0060,-0.1542,-0.2385,0.2863,-0.2098,-0.2807
lon,0.0974,-0.0141,0.0325,-0.1587,0.2137,0.2114,-0.1614,0.0918,-0.0612,0.4842,0.0617,-0.3978,0.0800,0.4137
alt,-0.1243,0.0320,-0.0809,-0.1351,-0.3443,0.2165,-0.0185,0.1865,-0.0612,0.0273,0.3929,0.0476,-0.0830,0.0038
roll,-0.1556,0.2064,-0.0739,-0.0448,-0.1440,-0.0652,-0.1763,-0.1513,0.1216,-0.1086,0.3693,0.2910,0.1718,0.0229
pitch,0.1876,0.0233,-0.2440,-0.0865,-0.0123,-0.0760,-0.2413,-0.1232,0.1097,0.2907,-0.1602,-0.0653,-0.3925,-0.2573
yaw,-0.0150,0.0384,0.0540,-0.0060,0.4489,0.2462,-0.1284,-0.0582,-0.0716,0.0808,0.0300,0.2409,-0.0433,-0.1384
vn,-0.1838,0.1809,0.0255,-0.1213,0.2391,0.2570,-0.0329,0.1022,-0.0932,0.1804,-0.1356,0.1383,0.0012,-0.0711
ve,-0.1127,0.1169,0.0557,-0.2808,0.2340,-0.2253,-0.0393,-0.2150,0.1010,-0.2197,0.0219,-0.0020,0.0233,-0.0237
vf,-0.2343,0.2033,-0.0433,-0.1172,-0.2091,0.0642,0.0916,0.1705,-0.0387,0.1032,-0.1658,0.0165,-0.0030,-0.1032
vl,0.0570,0.2121,0.0373,0.2683,0.1929,-0.0575,0.0138,0.0852,0.0539,-0.0008,-0.3330,0.0968,0.3503,0.0432



Largest absolute loadings for PC1:


,absolute_loading
posmode,0.268416
pos_accuracy,0.265881
velmode,0.256844
lat,0.235332
orimode,0.234667
speed_2d,0.234332
speed,0.234332
vf,0.234315
vel_accuracy,0.232753
yaw_rate_est,0.207320



Largest absolute loadings for PC2:


,absolute_loading
ay,0.347006
wz,0.317064
wu,0.316979
yaw_rate_est,0.316751
heading_change,0.316751
al,0.313207
vl,0.212107
roll,0.206379
speed,0.203372
speed_2d,0.203364


In [10]:
def evaluate_kmeans(features, cluster_counts):
    """Fit K-means for each K and return (metrics frame, {k: labels})."""
    rows = []
    labels_by_k = {}
    sample_size = min(SILHOUETTE_MAX_SAMPLES, len(features))

    for k in cluster_counts:
        if k >= len(features):
            warnings.warn(f"Skipping K={k}: insufficient observations.")
            continue

        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = model.fit_predict(features)

        if len(np.unique(labels)) < 2:
            score = np.nan
        else:
            score = silhouette_score(
                features,
                labels,
                sample_size=sample_size if sample_size < len(features) else None,
                random_state=RANDOM_STATE,
            )

        labels_by_k[k] = labels
        rows.append({"k": k, "silhouette_score": score, "inertia": model.inertia_})

    return pd.DataFrame(rows), labels_by_k


cluster_metrics, labels_by_k = evaluate_kmeans(X_pca, CLUSTER_COUNTS)

if cluster_metrics.empty or cluster_metrics["silhouette_score"].isna().all():
    raise ValueError("Unable to evaluate a valid K-means clustering.")

display(cluster_metrics.round(4))

,k,silhouette_score,inertia
0,3,0.2443,38464.9841
1,4,0.2776,33489.8927
2,5,0.3325,29024.3490


In [11]:
best_row = cluster_metrics.loc[cluster_metrics["silhouette_score"].idxmax()]
best_k = int(best_row["k"])
best_labels = labels_by_k[best_k]
analysis_df["cluster"] = best_labels

print(f"Selected K: {best_k}")
print(f"Silhouette score: {best_row['silhouette_score']:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cluster_metrics["k"], cluster_metrics["silhouette_score"], marker="o")
ax.set_xticks(cluster_metrics["k"])
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Silhouette score")
ax.set_title("K-means Model Comparison")
fig.tight_layout()
save_figure(fig, "kmeans_silhouette_scores.png")
plt.show()

Selected K: 5
Silhouette score: 0.3325


/var/folders/nk/l5mpcflj3z5396syzzlv9tw00000gn/T/ipykernel_64860/701114265.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
pca_cluster_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
pca_cluster_df["cluster"] = best_labels.astype(str)

fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(
    data=pca_cluster_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="tab10",
    s=25,
    alpha=0.75,
    linewidth=0,
    ax=ax,
)
ax.set_title(f"K-means Clusters in PCA Space (K={best_k})")
ax.legend(title="Cluster")
fig.tight_layout()
save_figure(fig, "pca_clusters.png")
plt.show()

display(
    analysis_df["cluster"].value_counts().sort_index().rename("row_count").to_frame()
)

/var/folders/nk/l5mpcflj3z5396syzzlv9tw00000gn/T/ipykernel_64860/3466040114.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,row_count
cluster,
0,92
1,271
2,207
3,383
4,483


In [13]:
missing_cols = [col for col in (X_COL, Y_COL) if col not in analysis_df.columns]
if missing_cols:
    raise KeyError(
        f"Trajectory columns not found: {missing_cols}. Update X_COL and Y_COL."
    )

sort_cols = [col for col in (DRIVE_COL, ORDER_COL) if col in analysis_df.columns]
trajectory_df = analysis_df.dropna(subset=[X_COL, Y_COL])
if sort_cols:
    trajectory_df = trajectory_df.sort_values(sort_cols)
trajectory_df = trajectory_df.assign(cluster=trajectory_df["cluster"].astype(str))

fig, ax = plt.subplots(figsize=(10, 8))

# Draw the path of each drive in order, then overlay the cluster labels.
if DRIVE_COL in trajectory_df.columns:
    paths = (group for _, group in trajectory_df.groupby(DRIVE_COL, sort=False))
else:
    paths = iter([trajectory_df])
for path in paths:
    ax.plot(path[X_COL], path[Y_COL], color="gray", alpha=0.35, linewidth=0.8)

sns.scatterplot(
    data=trajectory_df,
    x=X_COL,
    y=Y_COL,
    hue="cluster",
    hue_order=sorted(trajectory_df["cluster"].unique(), key=int),
    palette="tab10",
    s=25,
    alpha=0.8,
    linewidth=0,
    ax=ax,
)
ax.set_xlabel(X_COL)
ax.set_ylabel(Y_COL)
ax.set_title(f"Trajectory Coloured by Cluster (K={best_k})")
ax.set_aspect("equal", adjustable="datalim")
ax.legend(title="Cluster")
fig.tight_layout()
save_figure(fig, "trajectory_clusters.png")
plt.show()

/var/folders/nk/l5mpcflj3z5396syzzlv9tw00000gn/T/ipykernel_64860/4181324673.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
cluster_metrics.to_csv(OUTPUT_DIR / "cluster_metrics.csv", index=False)

pca_coordinates = pd.DataFrame(X_pca, columns=pc_cols)
pca_coordinates.insert(0, "row_index", analysis_df.index)
if DRIVE_COL in analysis_df.columns:
    pca_coordinates[DRIVE_COL] = analysis_df[DRIVE_COL].to_numpy()
pca_coordinates.to_csv(OUTPUT_DIR / "pca_coordinates.csv", index=False)

analysis_df.to_csv(OUTPUT_DIR / "data_with_clusters.csv", index=False)